# Demo 4 · Fine-tune Qwen3 into a decision model and publish it

**Part 05 · Fine-tuning** · 15 minutes + training

1. Generate the data: templates → scenarios → template split → near-duplicate filter → 2 orderings
2. Tokenize with the loss on the answer letter only, and check the targets
3. Train a LoRA adapter (Colab T4 or any recent GPU; Apple Silicon works, slower)
4. Score the fresh test set with the averaged letter scorer
5. Merge, write the model card, push to your Hugging Face account

**Colab:** Runtime → Change runtime type → T4 GPU, before running anything.

In [ ]:
%pip install -q -r requirements-finetune.txt

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

## 0 · Settings (the only cell you need to edit)

In [ ]:
BASE_MODEL = "Qwen/Qwen3-0.6B"              # or "Qwen/Qwen3-1.7B" with more time / GPU memory
HF_USERNAME = config.key("HF_USERNAME")     # set in .env or Colab secrets
REPO_NAME = "qwen3-0.6b-decision-router"
PUSH = True                                 # needs HF_TOKEN with write access
PRIVATE = True                              # make it public later, after reading your own model card

HP = dict(r=16, alpha=32, dropout=0.05, lr=2e-4, epochs=1, batch_size=4, grad_accum=2, max_length=512)

PREPARED_CHECKPOINT = ""   # backup model ID shared by the instructor if your training fails
print("device:", config.device(), "| repo:", f"{HF_USERNAME}/{REPO_NAME}")

## 1 · Data: templates → scenarios → split → filter → rows

In [ ]:
import json, time
import pandas as pd
from dmw import datagen, train as T, scorer as S, metrics
from dmw.prompt import build_messages

test_cases = datagen.load_test("data/fresh_test.json")
train_s, valid_s = datagen.generate()                                   # split by template
train_s, valid_s, dropped = datagen.drop_near_test(train_s, valid_s, [c["text"] for c in test_cases])
train_rows, valid_rows = datagen.to_rows(train_s), datagen.to_rows(valid_s, seed=7)
manifest = datagen.manifest(train_s, valid_s, train_rows, valid_rows, dropped)
print(json.dumps(manifest, indent=1))
print("max similarity of any training text to a test case:",
      round(datagen.max_similarity([s["text"] for s in train_s], [c["text"] for c in test_cases]), 2))

In [ ]:
pd.DataFrame(train_s)[["domain", "expected", "text"]].sample(8, random_state=1)

## 2 · Tokenize: same prompt function as the scorer, loss on the letter only

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
train_ex, skipped_tr = T.tokenize_rows(train_rows, tok, HP["max_length"])
valid_ex, skipped_va = T.tokenize_rows(valid_rows, tok, HP["max_length"])
print(f"train {len(train_ex)} (skipped {skipped_tr} over max_length) | valid {len(valid_ex)} (skipped {skipped_va})")

ex = train_ex[0]
print("\n--- end of one training example (the part with labels != -100 is learned) ---")
print(repr(tok.decode(ex["input_ids"][-40:])))
print("first trained tokens:", T.check_targets(train_ex, tok)[:10], "← all letters, never '<think>'")
lengths = pd.Series([len(e["input_ids"]) for e in train_ex])
print("tokens per example: median", int(lengths.median()), "| max", lengths.max())

## 3 · Model + LoRA

In [ ]:
model, tok, prec = T.prepare_model(BASE_MODEL, r=HP["r"], alpha=HP["alpha"], dropout=HP["dropout"])
model.print_trainable_parameters()
print("precision:", {k: v for k, v in prec.items() if k != "dtype"}, "| dtype:", prec["dtype"])

## 4 · Train  (start this early, then go back to the slides)

In [ ]:
OUT = config.OUTPUTS / "run"
trainer = T.make_trainer(model, tok, train_ex, valid_ex, OUT, prec, epochs=HP["epochs"], lr=HP["lr"],
                         batch_size=HP["batch_size"], grad_accum=HP["grad_accum"])
t0 = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - t0) / 60, 1)
print("training time:", TRAIN_MINUTES, "min")

In [ ]:
import matplotlib.pyplot as plt
tr, ev = T.loss_history(trainer)
plt.figure(figsize=(7, 3))
plt.plot(*zip(*tr), label="train loss", color="#075E91")
if ev: plt.plot(*zip(*ev), "o-", label="validation loss (held-out templates)", color="#C8641E")
plt.xlabel("step"); plt.legend(frameon=False); plt.tight_layout(); plt.show()

## 5 · Score the fresh test set (averaged over two orderings)

In [ ]:
model.config.use_cache = True
sc = S.LetterScorer(model, tok)
ft_rows = S.score_cases(sc, test_cases)
ft_summary = metrics.summarize(ft_rows)
T.save_json({"model": f"{HF_USERNAME}/{REPO_NAME}", "rows": ft_rows, "summary": ft_summary},
            config.OUTPUTS / "predictions_finetuned.json")
table = metrics.summary_table(ft_summary)
base_file = config.OUTPUTS / "baseline_base_model.json"
if base_file.exists():
    base = json.load(open(base_file))
    table["base accuracy"] = pd.Series({g: v["accuracy"] for g, v in base["summary"].items()}).round(3)
table

In [ ]:
wrong = [r for r in ft_rows if not r["correct"]]
pd.DataFrame([{**{k: r[k] for k in ("id", "group", "expected", "choice", "review")},
               "text": next(c["text"] for c in test_cases if c["id"] == r["id"])} for r in wrong])

## 6 · Merge, model card, push

In [ ]:
MODEL_DIR = config.OUTPUTS / "model"
T.merge_and_save(model, tok, MODEL_DIR)
hardware = config.device() if config.device() != "cuda" else __import__("torch").cuda.get_device_name(0)
card = T.model_card(f"{HF_USERNAME}/{REPO_NAME}", BASE_MODEL, manifest, HP, ft_summary,
                    hardware=f"{hardware}, {TRAIN_MINUTES} min")
print(card[:1200])

In [ ]:
if PUSH:
    url = T.push(MODEL_DIR, f"{HF_USERNAME}/{REPO_NAME}", card, private=PRIVATE)
    print("Published:", url, "(private)" if PRIVATE else "")
else:
    print("PUSH is False: the merged model is in", MODEL_DIR)

## If training failed

Set `PREPARED_CHECKPOINT` in the settings cell and run this, then continue with notebook 09.

In [ ]:
if PREPARED_CHECKPOINT:
    m2, t2 = S.load(PREPARED_CHECKPOINT)
    rows = S.score_cases(S.LetterScorer(m2, t2), test_cases)
    T.save_json({"model": PREPARED_CHECKPOINT, "rows": rows, "summary": metrics.summarize(rows)},
                config.OUTPUTS / "predictions_finetuned.json")
    print(metrics.summary_table(metrics.summarize(rows)))